# 🔦 08 · Scanning the beam: 4D-STEM, shadow images and depth sectioning
### Record a whole diffraction pattern at every probe position — then recover phase, magnified shadows and depth from one scan

**BiologyToResearch** · Part B — Imaging physics · Module 08 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/08_4dstem_shadow_tomography.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib |
| **Prerequisites** | [03 · Fourier transform](03_fourier_frequency.ipynb) (FFT, frequencies); [06 · Image formation](06_electron_image_formation.ipynb) (electron wavelength, weak phase object, defocus and the CTF); [07 · Tomography](07_tomography_missing_wedge.ipynb) (tilt series, depth of a specimen) |
| **Estimated time** | 110 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Seifer, Houben & Elbaum 2026 (shadow montage and cone-beam reconstruction in 4D-STEM tomography, *Microsc Microanal*); Seifer, Kirchweger, Edel & Elbaum 2024 (optimizing contrast in automated 4D-STEM cryo-tomography, *Microsc Microanal*); Kiriati, Fernández & Elbaum 2026 (*Tomo4D* and *Depth-resolved 4D-STEM by tomo4d*, conference abstracts) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods. The physics is real wave optics (200 kV
> electrons, apertures in milliradians, distances in nanometres and micrometres), but the specimens are a few thin
> layers of simple shapes, the only lens aberration is defocus, and there is no inelastic scattering. **All specimens and
> images are generated by code** (nothing is downloaded).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what scanning transmission electron microscopy (STEM) is, and what the "4D" in 4D-STEM means.
2. **Simulate** an electron probe and the diffraction pattern it makes in a thin specimen, and **form** virtual bright-field,
   annular and centre-of-mass (COM) images — including an integrated-COM (iCOM) image that shows the phase.
3. **Use** principal component analysis (PCA) on ring-shaped virtual detectors to separate kinds of contrast.
4. **Explain** why a *defocused* probe makes a magnified **shadow image** of the specimen, and **compute** its magnification.
5. **Assemble** a **shadow montage** with a synchronisation step $N_S$, use $N_S$ to **focus on different depths** of a thick
   specimen after the experiment, and **show** that the montage equals *tilt-corrected bright field*.
6. **Estimate** the depths of gold beads from a single scan, see how dose limits the precision, and **relate** all this to
   cone-beam 4D-STEM tomography.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools | 5 min |
| 1 · From pictures to scans | wavelength, aperture, a focused probe, a thin specimen | 15 min |
| 2 · Virtual detectors | BF, ABF, ADF, COM, iCOM; PCA of rings | 20 min |
| 3 · Defocus the probe: shadow images | geometry, magnification $L/\Delta f$ | 15 min |
| 4 · The shadow montage | patches → one image; the synchronisation step $N_S$ | 15 min |
| 5 · A thick specimen: focusing after the experiment | two layers, one scan | 10 min |
| 6 · Same sum, two orders: tilt-corrected bright field | parallax | 10 min |
| 7 · Depth from one scan | gold beads at six depths; dose | 10 min |
| 8 · Towards 4D-STEM tomography | cone beams, tilt series, tomo4d | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Cells are thick. A whole mammalian cell is several micrometres thick, and even its thin edges are 0.5–1 µm. The cryo-ET of
Modules 06–07 works best on specimens thinner than ~300 nm, which is why many labs thin cells with a focused ion beam.
**Scanning** transmission electron microscopy (STEM) offers another route: instead of illuminating the whole field and
forming an image with lenses *after* the specimen, a narrow beam — the **probe** — is scanned across the specimen, and a
detector simply counts electrons at each position. No lens after the specimen means no blur from electrons that lost
energy in a thick specimen, which is why Michael Elbaum's group at the Weizmann Institute pioneered cryo-STEM tomography
of whole, vitrified cells. Modern fast cameras can record the **entire diffraction pattern** at every probe position:
**4D-STEM** (two scan dimensions × two detector dimensions). Any detector can then be built *in the computer*, after the
experiment. The papers of this module show that the same 4D data hide a magnified **shadow image** of the specimen at
every probe position, and that these shadows reveal **depth** — a 3D view from a single scan.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
from numpy.fft import fft2, fftfreq, fftshift, ifft2
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

Two helpers from [`course_toolkit.py`](../course_toolkit.py): the relativistic electron wavelength (Module 06) and the
Pearson correlation (Module 04). Everything specific to 4D-STEM is written out in this notebook.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))
print("🧰 toolkit loaded")

---
## 1 · From pictures to scans

> 📍 **Where we are.** Module 06 described *conventional* transmission EM (TEM): a broad parallel beam, an objective lens
> that forms an image, defocus that turns phase into contrast. Module 07 tilted the specimen to get 3D. This module
> changes the illumination: a converging cone of electrons focused to a probe and **scanned**. Module 09 then goes back to
> TEM to build a whole in-silico cell.

### 1.1 Wavelength, aperture and probe

A condenser lens focuses the beam into a cone. An **aperture** limits the cone to a half-angle $\alpha$ (the
*convergence semi-angle*), typically a few **milliradians** (1 mrad = 0.001 rad ≈ 0.057°). Where the cone narrows to its
waist — the **crossover** — the probe is smallest: about $0.6\,\lambda/\alpha$ wide. Moving the specimen away from the
crossover by a distance $\Delta f$ (**defocus**) spreads the probe into a disc of radius $\approx\alpha\,\Delta f$.

In wave optics, the probe is the inverse Fourier transform of the aperture (a disc of radius $\alpha/\lambda$ in
frequency space), and defocus multiplies each frequency $k$ by the Fresnel phase $e^{-i\pi\lambda\Delta f k^2}$ — the
same $\chi$ that Module 06 used for the CTF.

In [ ]:
# ── 1.1 The electron probe ──
KV = 200
LAMBDA = electron_wavelength_A(KV) / 10          # wavelength in nm
PIXEL = 0.25                                     # specimen sampling, nm per pixel
ALPHA = 3e-3                                     # convergence semi-angle, rad (3 mrad)
print(f"{KV} kV electrons: wavelength λ = {LAMBDA * 1e3:.3f} pm; probe at focus ≈ 0.6 λ/α = {0.6 * LAMBDA / ALPHA:.2f} nm; "
      f"depth of field ≈ λ/α² = {LAMBDA / ALPHA ** 2:.0f} nm")


def k2_grid(n: int, pixel: float = PIXEL) -> np.ndarray:
    """|k|² on an unshifted n × n FFT grid (k in 1/nm)."""
    k = fftfreq(n, pixel)
    return k[:, None] ** 2 + k[None, :] ** 2


def fresnel(n: int, distance_nm: float) -> np.ndarray:
    """Free-space (Fresnel) propagation over distance_nm, as a multiplier in frequency space."""
    return np.exp(-1j * np.pi * LAMBDA * distance_nm * k2_grid(n)).astype(np.complex64)


def make_probe(n: int, defocus_nm: float = 0.0, alpha: float = ALPHA) -> np.ndarray:
    """Probe wave in an n × n window (centred), for a crossover defocus_nm above the specimen; Σ|ψ|² = 1.

    The aperture edge is smoothed over 10 % of α (as partial coherence does in a real microscope); a perfectly sharp
    edge would give ringing artefacts in this small simulation window."""
    edge = np.clip((alpha - np.sqrt(k2_grid(n)) * LAMBDA) / (0.1 * alpha) + 0.5, 0, 1)
    aperture = 0.5 - 0.5 * np.cos(np.pi * edge)
    psi = fftshift(ifft2(aperture * fresnel(n, defocus_nm)))
    return (psi / np.sqrt((np.abs(psi) ** 2).sum())).astype(np.complex64)


fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
for ax, df, half_px in zip(axes, [0.0, 1000.0, 3500.0], [12, 96, 96]):
    probe = np.abs(make_probe(192, df)[96 - half_px:96 + half_px, 96 - half_px:96 + half_px]) ** 2
    half = half_px * PIXEL
    ax.imshow(probe, cmap="magma", extent=[-half, half, half, -half])
    ax.set_title(f"probe intensity, Δf = {df / 1000:.1f} µm" + (f"\n(disc radius α·Δf = {ALPHA * df:.1f} nm)" if df else "\n(focused — note the 8× zoom)"),
                 fontsize=9)
    ax.set_xlabel("nm")
plt.tight_layout(); plt.show()

### 1.2 A thin specimen, and what the detector sees

As in Module 06, a thin biological specimen mostly shifts the **phase** of the electron wave: the wave leaving it is
$\psi_{\rm exit}(\mathbf r)=\psi_{\rm probe}(\mathbf r-\mathbf r_s)\,t(\mathbf r)$ with the **transmission function**
$t=e^{i\phi(\mathbf r)-\mu(\mathbf r)}$ ($\phi$: phase shift, $\mu$: a little absorption, mainly from heavy atoms). The
detector sits far below the specimen, so it records the **far-field diffraction pattern** — the squared Fourier transform:

$$I(\mathbf k;\mathbf r_s)=\big|\mathcal F[\psi_{\rm probe}(\mathbf r-\mathbf r_s)\,t(\mathbf r)]\big|^2 .$$

*In words:* for each probe position $\mathbf r_s$ (2 numbers) we record a whole picture over detector positions
$\mathbf k$ (2 numbers) — four dimensions. A detector pixel $\mathbf k$ corresponds to a scattering **angle**
$\theta=\lambda k$. The unscattered beam fills a disc of radius $\alpha$: the **bright-field (BF) disc**.

Our first specimen is a thin layer with a membrane vesicle and a few proteins (pure phase) and two gold beads (strong phase
plus absorption).

In [ ]:
# ── 1.2 A thin specimen ──
def gaussian_spot(shape, centre_nm, sigma_nm):
    """A 2D Gaussian (peak 1) centred at centre_nm = (y, x) on a grid of PIXEL-nm pixels."""
    yy, xx = np.indices(shape) * PIXEL
    return np.exp(-((yy - centre_nm[0]) ** 2 + (xx - centre_nm[1]) ** 2) / (2 * sigma_nm ** 2))


def ring(shape, centre_nm, radius_nm, width_nm=1.0):
    """A thin ring (a membrane seen from above) of peak 1."""
    yy, xx = np.indices(shape) * PIXEL
    r = np.hypot(yy - centre_nm[0], xx - centre_nm[1])
    return np.exp(-(r - radius_nm) ** 2 / (2 * width_nm ** 2))


def transmission(phase, absorption, kmax=0.8):
    """t = exp(iφ − μ), smoothly band-limited to |k| ≤ kmax (1/nm) so the simulation grid never aliases."""
    t = np.exp(1j * phase - absorption)
    taper = np.clip((kmax - np.sqrt(k2_grid(t.shape[0]))) / (0.25 * kmax), 0, 1)
    return ifft2(fft2(t) * (0.5 - 0.5 * np.cos(np.pi * taper))).astype(np.complex64)


THIN = (192, 192)                                                       # 48 nm × 48 nm
bio_phase = 0.25 * ring(THIN, (23, 23), 8.0)                           # membrane
for c in [(15, 32), (32, 14), (31, 31), (23, 23), (16, 13)]:
    bio_phase += 0.3 * gaussian_spot(THIN, c, 1.5)                     # proteins
gold = gaussian_spot(THIN, (34, 24), 0.8) + gaussian_spot(THIN, (12, 22), 0.8)
thin_phase = bio_phase + 1.0 * gold
thin_absorb = 0.4 * gold
t_thin = transmission(thin_phase, thin_absorb)
print(f"phase shift: up to {thin_phase.max():.2f} rad (gold), {0.3:.2f} rad (protein); mean |t| = {np.abs(t_thin).mean():.3f}")

---
## 2 · Virtual detectors

We scan a **focused** probe over the specimen in 0.5 nm steps and record a 64 × 64-pixel pattern at each of 60 × 60
positions: 3,600 patterns, about 60 MB. Each pattern is normalised so that it sums to 1 (it is the probability
distribution of where one electron lands).

In [ ]:
# ── 2.1 Record a 4D dataset with a focused probe ──
def scan_4d(layers, gaps_nm, probe, step_px, n_scan, offset_px=0, crop=None):
    """4D data I[iy, ix, ky, kx]: the probe crosses thin layers separated by gaps (multislice), far-field intensity.

    layers: transmission functions (same shape); gaps_nm: distances between consecutive layers; crop: keep the central
    2·crop × 2·crop detector pixels."""
    n = probe.shape[0]
    props = [fresnel(n, g) for g in gaps_nm]
    rows = []
    for iy in range(n_scan):
        y0 = offset_px + iy * step_px
        windows = [np.stack([L[y0:y0 + n, offset_px + ix * step_px: offset_px + ix * step_px + n] for ix in range(n_scan)])
                   for L in layers]
        psi = probe[None] * windows[0]
        for prop, w in zip(props, windows[1:]):                             # propagate to the next layer, transmit
            psi = ifft2(fft2(psi) * prop) * w
        pattern = np.abs(fftshift(fft2(psi, norm="ortho"), axes=(-2, -1))) ** 2
        if crop:
            pattern = pattern[:, n // 2 - crop:n // 2 + crop, n // 2 - crop:n // 2 + crop]
        rows.append(pattern.astype(np.float32))
    return np.stack(rows)


def detector_angles(n_window: int, size: int):
    """Scattering angles (θy, θx) in rad of a size × size (centred) detector for an n_window-pixel simulation window."""
    dtheta = LAMBDA / (n_window * PIXEL)                                    # angle per detector pixel
    c = (np.arange(size) - size // 2) * dtheta
    return np.meshgrid(c, c, indexing="ij")


NF, STEP_F, NSCAN_F = 64, 2, 60                                           # window, step (pixels), scan positions
t0 = time.time()
data_focus = scan_4d([t_thin], [], make_probe(NF), STEP_F, NSCAN_F)
TY, TX = detector_angles(NF, NF)
THETA = np.hypot(TY, TX)
print(f"4D dataset {data_focus.shape} ({data_focus.nbytes / 1e6:.0f} MB) in {time.time() - t0:.1f} s; "
      f"detector: {LAMBDA / (NF * PIXEL) * 1e3:.3f} mrad per pixel, BF disc radius {ALPHA / (LAMBDA / (NF * PIXEL)):.1f} pixels")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
ext = [TX.min() * 1e3, TX.max() * 1e3, TY.max() * 1e3, TY.min() * 1e3]
for ax, (iy, ix, name) in zip(axes, [(5, 5, "empty ice"), (46, 46, "on a protein"), (52, 32, "on a gold bead")]):
    ax.imshow(np.log10(data_focus[iy, ix] + 1e-9), vmin=-9, vmax=-3, cmap="magma", extent=ext)
    ax.set_title(f"pattern at probe position {name}\n(log scale)", fontsize=9); ax.set_xlabel("θx (mrad)")
plt.tight_layout(); plt.show()

🔍 The patterns look almost identical: a bright disc of radius α = 3 mrad (the unscattered beam) and, on a log scale, a
faint halo of electrons scattered outside it. The information about the specimen is in **small redistributions**
inside the disc and in the halo. Which ones? We decide *after* the experiment by summing the patterns over chosen
regions — **virtual detectors**:

- **BF** (bright field): the inner part of the disc, $\theta<\alpha/2$; **ABF** (annular bright field): $\alpha/2<\theta<\alpha$;
- **ADF** (annular dark field): outside the disc, $\theta>1.15\,\alpha$ — only scattered electrons;
- **COM** (centre of mass): where, on average, the electron landed, $\langle\boldsymbol\theta\rangle=\sum_k\boldsymbol\theta_k I_k$.

A phase gradient tilts the wave, so the electrons are deflected: $\langle\boldsymbol\theta\rangle=\frac{\lambda}{2\pi}\nabla\phi$
(smoothed by the probe). The COM is therefore a map of the **slope** of the phase, and integrating it — **iCOM**, done in
Fourier space as in Module 03 — gives back the **phase** itself.

In [ ]:
# ── 2.2 Virtual detectors, COM and iCOM ──
def virtual_image(data, mask):
    """Sum every pattern over the detector pixels selected by mask."""
    return np.tensordot(data, mask.astype(np.float32), axes=([2, 3], [0, 1]))


bf = virtual_image(data_focus, THETA < ALPHA / 2)
abf = virtual_image(data_focus, (THETA >= ALPHA / 2) & (THETA < ALPHA))
adf = virtual_image(data_focus, THETA > 1.15 * ALPHA)
total = data_focus.sum((2, 3))
com_y, com_x = virtual_image(data_focus, TY) / total, virtual_image(data_focus, TX) / total   # rad


def integrate_com(com_y, com_x, step_nm):
    """Phase (rad) from the COM deflection: solve ∇φ = (2π/λ)·COM in Fourier space."""
    ky = fftfreq(com_y.shape[0], step_nm)
    KY, KX = np.meshgrid(ky, ky, indexing="ij")
    k2 = KY ** 2 + KX ** 2
    k2[0, 0] = 1.0
    phi = (KY * fft2(com_y) + KX * fft2(com_x)) / (1j * LAMBDA * k2)
    phi[0, 0] = 0.0
    return np.real(ifft2(phi))


icom = integrate_com(com_y, com_x, STEP_F * PIXEL)
c0 = NF // 2                                                             # probe centre within the first window
grid = (slice(c0, c0 + NSCAN_F * STEP_F, STEP_F),) * 2                # specimen pixels under the probe centres
true_phase, true_bio, true_gold = thin_phase[grid], bio_phase[grid], gold[grid]
images = {"true phase φ": true_phase, "BF": bf, "ABF": abf, "ADF": adf, "COM x (mrad)": com_x * 1e3, "iCOM (rad)": icom}
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for ax, (name, im) in zip(axes, images.items()):
    ax.imshow(im); ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()
print("correlation with:   whole phase   proteins+membrane   gold")
for name, im in list(images.items())[1:]:
    print(f"{name:13s}   {pearson(im, true_phase):+10.2f}   {pearson(im, true_bio):+16.2f}   {pearson(im, true_gold):+6.2f}")
print(f"iCOM range {icom.max() - icom.min():.2f} rad vs true phase range {true_phase.max() - true_phase.min():.2f} rad")

🔍 **What just happened?** With a focused probe, the BF and ABF images show only the **gold** (correlation ≈ −1 with
the gold, ≈ 0 with the proteins and membrane): an in-focus image of a weak *phase* object has almost no intensity
contrast — Module 06's lesson again. The ADF image also shows only the gold, which scatters electrons far outside the
disc. The **COM** image shows the membrane and proteins as *slopes* — light on one side, dark on the other (so its
correlation with the phase itself is zero) — and **iCOM** turns them back into the phase, in radians: correlation ≈ 1
with the true phase map, with the gold peak only slightly lower than the truth because the probe smooths it. Only a
pixelated detector, recording the whole pattern, makes COM possible.

### 2.3 PCA of ring detectors

Seifer, Kirchweger, Edel & Elbaum (2024) go further: instead of choosing detectors by hand, cut the detector into many
thin **rings** and let **PCA** (Module 04's idea of finding the directions of largest variation) find which combinations
of rings carry independent contrast. Each ring gives an image; PCA treats the 12 ring images as 12 measurements per scan
position.

In [ ]:
# ── 2.3 PCA of 12 ring-shaped virtual detectors ──
edges = np.linspace(0, 4.5e-3, 13)
ring_images = np.stack([virtual_image(data_focus, (THETA >= a) & (THETA < b)) for a, b in zip(edges[:-1], edges[1:])])
X = ring_images.reshape(12, -1)
X = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)              # each ring: mean 0, sd 1
U, svals, Vt = np.linalg.svd(X, full_matrices=False)
explained = svals ** 2 / (svals ** 2).sum()
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
centres = (edges[:-1] + edges[1:]) / 2 * 1e3
for i in range(3):
    axes[i].imshow(Vt[i].reshape(NSCAN_F, NSCAN_F)); axes[i].axis("off")
    axes[i].set_title(f"PC{i + 1} ({explained[i]:.1%} of variance)\nr(proteins+membrane) {pearson(Vt[i], true_bio):+.2f}, "
                      f"r(gold) {pearson(Vt[i], true_gold):+.2f}", fontsize=8)
    axes[3].plot(centres, U[:, i], "o-", label=f"PC{i + 1}")
axes[3].axvline(ALPHA * 1e3, color="gray", ls=":"); axes[3].text(ALPHA * 1e3, axes[3].get_ylim()[1] * 0.9, " α", fontsize=8)
axes[3].set_xlabel("ring radius (mrad)"); axes[3].set_title("weights of each ring", fontsize=9); axes[3].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** PC1 (almost all of the variance) weights all bright-field rings against the dark-field rings:
it is the **absorption/scattering** image — the gold. A later component, although it carries less than 1 % of the
variance, matches the **proteins and membrane** (|r| ≈ 0.75; the sign of a component is arbitrary) and is dominated by
the ring just *outside* the edge of the disc (θ ≈ α): a phase gradient tilts the beam slightly, moving electrons across
the disc edge. PCA has separated scattering-type and phase-type contrast without being told what either is. Two lessons:
a small share of the variance can carry the biology, and which component comes second or third depends on the
specimen — look at the ring weights, not only the order.

✏️ **Your turn.** Make the probe 10× wider in angle (`ALPHA = 0.3e-3`) — a nearly parallel beam — and rerun Sections 1–2.
What happens to the BF disc, the COM and iCOM images, and the sharpness?

---
## 3 · Defocus the probe: shadow images

Now move the crossover **above** the specimen by a few micrometres. The probe becomes a wide disc, and something
surprising appears inside the BF disc of each pattern: a **picture** of the specimen. The geometry is that of a shadow
puppet: the crossover acts as a point light source, the specimen casts a shadow, and the shadow spreads as it travels
the camera length $L$ to the detector. A point of the specimen at distance $r$ from the beam axis is seen at angle
$\theta=r/\Delta f$ and lands at $L\theta$ on the detector, so the shadow is magnified by

$$M=\frac{L}{\Delta f}.$$

*In words:* the closer the crossover is to the specimen, the stronger the magnification. With $L=1.7$ m and
$\Delta f=3.5$ µm, $M\approx 500{,}000\times$. In our simulation the detector is described by its angle per pixel
$\delta\theta=\lambda/(N\,p)$ ($N$ pixels of size $p$ in the simulation window), so one detector pixel corresponds to
$\Delta f\,\delta\theta$ nanometres of specimen — the **shadow pixel**.

In [ ]:
# ── 3.1 The geometry, drawn to scale in angle (not in length) ──
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot([-3, 0, 3], [6, 4, 6], color="tab:blue"); ax.text(3.1, 6, "converging beam\n(from the condenser lens)", fontsize=8, va="center")
ax.plot(0, 4, "o", color="tab:red"); ax.text(0.25, 4, "crossover (point source)", fontsize=8, va="center")
ax.fill_between([-4, 4], 2.4, 2.6, color="0.8"); ax.text(4.1, 2.5, "specimen, Δf below", fontsize=8, va="center")
ax.plot(-0.6, 2.5, "s", color="k", ms=5)
ax.plot([0, -3], [4, -1], color="tab:blue", lw=0.8); ax.plot([0, 3], [4, -1], color="tab:blue", lw=0.8)
ax.plot([0, -2.0], [4, -1], color="k", ls="--", lw=0.8); ax.plot(-2.0, -1, "s", color="k", ms=10)
ax.fill_between([-4, 4], -1.15, -0.85, color="tab:orange", alpha=0.5); ax.text(4.1, -1, "detector, L below", fontsize=8, va="center")
ax.annotate("", xy=(-3.6, 2.5), xytext=(-3.6, 4), arrowprops=dict(arrowstyle="<->")); ax.text(-3.55, 3.25, " Δf", fontsize=9)
ax.annotate("", xy=(-3.9, -1), xytext=(-3.9, 4), arrowprops=dict(arrowstyle="<->")); ax.text(-3.85, 1.0, " L", fontsize=9)
ax.text(-1.95, -0.55, "magnified shadow\nof the particle", fontsize=8)
ax.set_xlim(-4.2, 6.5); ax.set_ylim(-1.6, 6.4); ax.axis("off")
ax.set_title("Shadow imaging: a defocused probe acts as a point source (not to scale)", fontsize=9)
plt.tight_layout(); plt.show()

Let us see it. We use a 48 nm simulation window ($N=192$ pixels of 0.25 nm), keep the central 128 × 128 detector
pixels, and record one pattern at the same probe position for four defocus values.

In [ ]:
# ── 3.2 Patterns at one probe position for increasing defocus ──
N, CROP = 192, 64
DTHETA = LAMBDA / (N * PIXEL)
DISC = (np.hypot(*detector_angles(N, 2 * CROP)) <= ALPHA - DTHETA)         # bright-field disc (minus its edge pixel)

TOP = (456, 456)                                                          # 114 nm × 114 nm specimen layers
r_top = np.random.default_rng(1)
top_phase = 0.25 * ring(TOP, (52, 60), 12.0, 1.2)
for _ in range(14):
    top_phase += 0.3 * gaussian_spot(TOP, r_top.uniform(30, 84, 2), r_top.uniform(1.2, 2.0))
t_top = transmission(top_phase, np.zeros(TOP))

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
y0 = x0 = 140
axes[0].imshow(top_phase[y0:y0 + N, x0:x0 + N], extent=[0, N * PIXEL, N * PIXEL, 0])
axes[0].add_patch(plt.Circle((N * PIXEL / 2, N * PIXEL / 2), ALPHA * 3500, fill=False, color="tab:orange"))
axes[0].set_title("specimen (phase); circle: probe\nat Δf = 3.5 µm", fontsize=9); axes[0].set_xlabel("nm")
for ax, df in zip(axes[1:], [0.0, 500.0, 1500.0, 3500.0]):
    pat = scan_4d([t_top], [], make_probe(N, df), 1, 1, offset_px=y0, crop=CROP)[0, 0]
    ax.imshow(np.where(DISC, pat, np.nan)); ax.axis("off")
    ax.set_title(f"BF disc, Δf = {df / 1000:.1f} µm\n(shadow pixel {df * DTHETA:.2f} nm)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 At focus the disc is nearly uniform. As the defocus grows, the disc fills with a picture: at 3.5 µm the bright curved
band is the vesicle membrane crossing the orange circle (the probe's footprint on the specimen, radius
$\alpha\Delta f$ ≈ 10 nm), magnified so that one detector pixel covers only ~0.18 nm of specimen. The contrast comes
from **Fresnel diffraction**: between the specimen and the far field, the phase shifts turn into intensity exactly as
defocus does in Module 06.

**Check the theory.** If the pattern really is a shadow, it should equal the Fresnel image of the specimen at defocus
$\Delta f$, sampled at the points $\Delta f\,\boldsymbol\theta$:

In [ ]:
# ── 3.3 The pattern equals the magnified Fresnel image of the specimen ──
DF = 3500.0
window = t_top[y0:y0 + N, x0:x0 + N]
fresnel_image = np.abs(ifft2(fft2(window) * fresnel(N, DF))) ** 2           # plane wave, propagated Δf
TYc, TXc = detector_angles(N, 2 * CROP)
coords = np.array([N / 2 + DF * TYc / PIXEL, N / 2 + DF * TXc / PIXEL])    # specimen pixel seen by each detector pixel
predicted = ndimage.map_coordinates(fresnel_image, coords, order=1)
pattern = scan_4d([t_top], [], make_probe(N, DF), 1, 1, offset_px=y0, crop=CROP)[0, 0]
inner = np.hypot(TYc, TXc) <= 0.8 * ALPHA
print(f"correlation between simulated pattern and magnified Fresnel image: {pearson(pattern[DISC], predicted[DISC]):.3f} "
      f"(whole BF disc), {pearson(pattern[inner], predicted[inner]):.3f} (central 80 % of its radius)")
print(f"magnification for a camera length of 1.74 m: L/Δf = {1.74 / (DF * 1e-9):,.0f}×")

Away from its edge the disc matches the prediction perfectly (correlation 1.000): the BF disc of a defocused probe *is* a
magnified, defocused (Fresnel) image of the specimen. (Near the edge, the smoothed aperture dims the pattern — something
the plane-wave prediction does not include — hence the lower value for the whole disc.) ✏️
**Your turn.** Put the crossover *below* the specimen (`make_probe(N, -DF)`) and redo 3.2. What happens to the orientation of
the shadow and to its contrast? (Hint: rays cross before reaching the specimen.)

---
## 4 · The shadow montage

Each pattern shows only a 20 nm patch. Scanning the probe in steps of $s$ moves the patch over the specimen by $s$ —
which is $s/(\Delta f\,\delta\theta)$ shadow pixels. Seifer, Houben & Elbaum (2026) assemble a **shadow montage**: place
each patch in a larger, finer grid, shifted by $N_S$ grid pixels per scan step, add them up, and divide by how many
patches covered each grid pixel. The **synchronisation step** $N_S$ is correct when it matches the shift of the shadow:

$$N_S=\frac{s}{\Delta f\,\delta\theta},\qquad\text{montage pixel}=\frac{s}{N_S}.$$

*In words:* the montage is like assembling a panorama from overlapping photos — you must shift each photo by exactly the
right amount. We choose $s=2$ nm and put the crossover where the top of our specimen needs $N_S=10$ exactly
($\Delta f\approx3.83$ µm), and scan 32 × 32 positions.

In [ ]:
# ── 4.1 Scan the top layer with a defocused probe and assemble montages ──
STEP, NSCAN, OFFSET = 8, 32, 8                                            # 8 px = 2 nm per scan step
S_NM = STEP * PIXEL
DF_TOP = S_NM / (10 * DTHETA)                                             # crossover distance for N_S = 10
RS0 = OFFSET * PIXEL + N // 2 * PIXEL                                     # specimen position of the first probe centre (nm)
t0 = time.time()
data_top = scan_4d([t_top], [], make_probe(N, DF_TOP), STEP, NSCAN, OFFSET, crop=CROP)
print(f"Δf = {DF_TOP:.0f} nm; {NSCAN}×{NSCAN} patterns of {2 * CROP}×{2 * CROP} pixels in {time.time() - t0:.1f} s")


def shadow_montage(data, ns_sync, disc=DISC):
    """Shadow montage: put each BF disc ns_sync grid pixels further per scan step; average overlaps."""
    n, c = data.shape[0], data.shape[2]
    size = int(np.ceil((n - 1) * ns_sync)) + c
    acc, cnt, w = np.zeros((size, size)), np.zeros((size, size)), disc.astype(float)
    for iy in range(n):
        oy = int(round(iy * ns_sync))
        for ix in range(n):
            ox = int(round(ix * ns_sync))
            acc[oy:oy + c, ox:ox + c] += data[iy, ix] * w
            cnt[oy:oy + c, ox:ox + c] += w
    return acc / np.maximum(cnt, 1), cnt


def on_montage_grid(layer_map, size, ns_sync):
    """Resample a specimen map onto the grid of a montage with synchronisation step ns_sync (montage pixel s/N_S)."""
    pos = (np.arange(size) - CROP) * S_NM / ns_sync + RS0                  # specimen coordinate (nm) of each grid pixel
    return ndimage.map_coordinates(layer_map, np.meshgrid(pos / PIXEL, pos / PIXEL, indexing="ij"), order=1)


def show_montage(ax, img, cnt, title):
    """Display the well-covered part of a montage."""
    keep = cnt >= 0.6 * cnt.max()
    rows, cols = np.where(keep.any(1))[0], np.where(keep.any(0))[0]
    sub = img[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    ax.imshow(sub, vmin=np.percentile(sub, 1), vmax=np.percentile(sub, 99)); ax.set_title(title, fontsize=9); ax.axis("off")


fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
m10, cnt10 = shadow_montage(data_top, 10)
truth = on_montage_grid(top_phase, m10.shape[0], 10)
show_montage(axes[0], truth, cnt10, "specimen (phase)")
for ax, ns in zip(axes[1:], [8, 10, 12]):
    m, cnt = shadow_montage(data_top, ns)
    show_montage(ax, m, cnt, f"montage, N_S = {ns}" + (" ✓" if ns == 10 else ""))
plt.tight_layout(); plt.show()
print(f"montage pixel at N_S = 10: {S_NM / 10:.2f} nm; scan field {(NSCAN - 1) * S_NM:.0f} nm + one patch "
      f"({2 * ALPHA * DF_TOP:.0f} nm across)")

🔍 **What just happened?** With the right synchronisation step the patches fit together into one sharp, magnified image of
the whole scanned area — fine detail (0.2 nm montage pixels) from a scan with 2 nm steps. With the wrong $N_S$, every
feature is placed slightly differently by each overlapping patch, and the montage blurs or doubles. (The dark and bright
fringes at edges are Fresnel phase contrast, the defocus "CTF" of Module 06; the paper corrects it on the assembled
montage.)

---
## 5 · A thick specimen: focusing after the experiment

The key observation: the magnification $L/\Delta f$ depends on the distance from the crossover. A layer deeper in the
specimen is farther from the crossover, so its shadow is **less** magnified and moves by **fewer** shadow pixels per scan
step — it needs a **smaller** $N_S$. One scan therefore contains a **focal series**: choosing $N_S$ after the experiment
selects a depth. The depth that one step of $N_S$ moves the focus is about

$$\delta z\approx\frac{\Delta f}{N_S}.$$

We add a second layer (filaments and gold beads) 0.96 µm below the first — the bottom of a thick cell section, positioned
so that it needs $N_S = 8$.

In [ ]:
# ── 5.1 Two layers, one scan ──
DF_BOTTOM = S_NM / (8 * DTHETA)
GAP = DF_BOTTOM - DF_TOP
r_bot = np.random.default_rng(2)
bottom_phase, bottom_absorb = np.zeros(TOP), np.zeros(TOP)
yy, xx = np.indices(TOP) * PIXEL
for angle, offset in [(0.5, -15), (2.0, 5), (1.1, 22)]:                  # three straight filaments
    dist = (yy - 57) * np.cos(angle) - (xx - 57) * np.sin(angle) - offset
    bottom_phase += 0.25 * np.exp(-dist ** 2 / (2 * 1.0 ** 2))
for _ in range(4):                                                        # four gold beads
    g = gaussian_spot(TOP, r_bot.uniform(32, 82, 2), 1.2)
    bottom_phase += 1.0 * g; bottom_absorb += 0.3 * g
t_bottom = transmission(bottom_phase, bottom_absorb)

t0 = time.time()
data_two = scan_4d([t_top, t_bottom], [GAP], make_probe(N, DF_TOP), STEP, NSCAN, OFFSET, crop=CROP)
print(f"bottom layer {GAP:.0f} nm below the top; scan in {time.time() - t0:.1f} s; depth step per unit of N_S at "
      f"N_S = 10: Δf/N_S ≈ {DF_TOP / 10:.0f} nm (depth of field λ/α² = {LAMBDA / ALPHA ** 2:.0f} nm)")

m_top, c_top = shadow_montage(data_two, 10)
m_bot, c_bot = shadow_montage(data_two, 8)
fig, axes = plt.subplots(2, 3, figsize=(11, 7.2))
show_montage(axes[0, 0], on_montage_grid(top_phase, m_top.shape[0], 10), c_top, "top layer (truth)")
show_montage(axes[0, 1], on_montage_grid(bottom_phase, m_bot.shape[0], 8), c_bot, "bottom layer (truth)")
show_montage(axes[0, 2], on_montage_grid(top_phase + bottom_phase, m_top.shape[0], 10), c_top,
             "both, superimposed\n(what a projection would show)")
show_montage(axes[1, 0], m_top, c_top, "montage N_S = 10 → top in focus")
show_montage(axes[1, 1], m_bot, c_bot, "montage N_S = 8 → bottom in focus")
bf_two = virtual_image(data_two, DISC)
axes[1, 2].imshow(bf_two); axes[1, 2].axis("off"); axes[1, 2].set_title("virtual BF image (whole disc)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The same 1,024 patterns give two different pictures: with $N_S=10$ the vesicle and proteins
are sharp and the filaments are smeared; with $N_S=8$ the filaments and gold beads are sharp and the vesicle is a blur.
The ordinary virtual BF image — the sum of each pattern — is blurred by the 20-nm probe and mixes everything. Depth
information is recorded in *where inside the disc* each shadow falls. The out-of-focus blur has the size of the cone at
the wrong depth, about $2\alpha\,\Delta z$ ≈ 6 nm here.

In [ ]:
# ── 5.2 A focal series from one scan ──
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for ax, ns in zip(axes, [7, 8, 9, 10, 11, 12]):
    m, cnt = shadow_montage(data_two, ns)
    z = S_NM / (ns * DTHETA) - DF_TOP
    show_montage(ax, m, cnt, f"N_S = {ns}\n(focus {abs(z):.0f} nm {'below' if z >= 0 else 'above'} top)")
plt.tight_layout(); plt.show()

---
## 6 · Same sum, two orders: tilt-corrected bright field

Another way to read 4D-STEM data: pick **one** detector pixel and make an image from it across all scan positions. Each
detector pixel sees the specimen along a slightly different direction — like your two eyes — so these images are
**shifted** relative to each other (**parallax**), by an amount proportional to the depth. Shifting them back and adding
them up is called **tilt-corrected bright field (tcBF)**. Seifer, Houben & Elbaum point out that this is the *same sum* as
the shadow montage, only in a different order: montage = loop over probe positions, paste each pattern; tcBF = loop over
detector pixels, paste each image.

In [ ]:
# ── 6.1 tcBF as a loop over detector pixels — identical to the montage ──
def tcbf(data, ns_sync, disc=DISC):
    """Tilt-corrected bright field: each detector pixel's scan image, shifted by its parallax, averaged."""
    n, c = data.shape[0], data.shape[2]
    size = int(np.ceil((n - 1) * ns_sync)) + c
    acc, cnt = np.zeros((size, size)), np.zeros((size, size))
    grid = np.round(np.arange(n) * ns_sync).astype(int)                    # where each scan step lands
    for ky, kx in zip(*np.nonzero(disc)):
        idx = np.ix_(ky + grid, kx + grid)
        acc[idx] += data[:, :, ky, kx]                                     # one virtual image, shifted by (ky, kx)
        cnt[idx] += 1
    return acc / np.maximum(cnt, 1)


t0 = time.time()
tc = tcbf(data_two, 10)
print(f"tcBF in {time.time() - t0:.1f} s; largest difference from the shadow montage: {np.abs(tc - m_top).max():.1e}")

# ── 6.2 Parallax: two detector pixels, two depths ──
data_bottom_only = scan_4d([t_bottom], [], make_probe(N, DF_BOTTOM), STEP, NSCAN, OFFSET, crop=CROP)


def shift_between(a, b):
    """Integer shift (rows, cols) that best aligns image b onto image a (FFT cross-correlation, Module 03)."""
    xc = np.real(ifft2(fft2(a - a.mean()) * np.conj(fft2(b - b.mean()))))
    s = np.array(np.unravel_index(np.argmax(xc), xc.shape))
    return np.where(s > a.shape[0] // 2, s - a.shape[0], s)


left, right = (CROP, CROP - 40), (CROP, CROP + 40)                       # two detector pixels 80 pixels apart
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for row, (name, d, ns) in enumerate([("top layer", data_top, 10), ("bottom layer", data_bottom_only, 8)]):
    a, b = d[:, :, left[0], left[1]], d[:, :, right[0], right[1]]
    shift = shift_between(a, b)
    axes[2 * row].imshow(a); axes[2 * row + 1].imshow(b)
    axes[2 * row].set_title(f"{name}: detector pixel left", fontsize=9); axes[2 * row + 1].set_title(f"{name}: right", fontsize=9)
    print(f"{name:12s}: measured parallax {abs(shift[1])} scan steps; predicted 80/N_S = {80 / ns:.0f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The two orders of summation agree to rounding error: shadow montage = tcBF. And the parallax
between two detector pixels is larger for the deeper layer (10 scan steps) than for the top layer (8): exactly the depth
cue of stereo vision. This is why the authors describe 4D-STEM with a defocused probe as **cone-beam** imaging: every
detector pixel is a ray from the crossover, and the montage is a back-projection along those rays (Module 07).

---
## 7 · Depth from one scan

The titles of two 2026 abstracts by Kiriati, Fernández & Elbaum speak of **depth-resolved 4D-STEM**. Let us ask a
concrete version: from a single scan, how well can we place small gold beads in depth? We put 12 beads in six thin
layers 250 nm apart (a 1.25 µm thick specimen), scan, compute montages for a fine range of $N_S$, and for each bead find
the $N_S$ at which its spot is most contrasted. Then depth $=s/(N_S\,\delta\theta)-\Delta f_{\rm top}$.

In [ ]:
# ── 7.1 Twelve beads at six depths ──
DEPTHS = np.array([0, 250, 500, 750, 1000, 1250.0])
DF_BEADS = S_NM / (11 * DTHETA)                                           # the top layer needs N_S = 11
r_b = np.random.default_rng(3)
beads, bead_layers = [], []
lo, hi = RS0 + 3, RS0 + (NSCAN - 1) * S_NM - 3
for z in DEPTHS:
    ph, ab = np.zeros(TOP), np.zeros(TOP)
    for _ in range(2):
        for _try in range(1000):                                          # keep beads ≥ 12 nm apart
            c = r_b.uniform(lo, hi, 2)
            if all(np.hypot(*(c - np.array(b[1:]))) > 12 for b in beads):
                break
        g = gaussian_spot(TOP, c, 0.8)
        ph += 1.0 * g; ab += 0.4 * g
        beads.append((z, *c))
    bead_layers.append(transmission(ph, ab))
beads = np.array(beads)
t0 = time.time()
data_beads = scan_4d(bead_layers, np.diff(DEPTHS), make_probe(N, DF_BEADS), STEP, NSCAN, OFFSET, crop=CROP)
print(f"scan through 6 layers in {time.time() - t0:.1f} s")

NS_GRID = np.round(np.arange(7.0, 12.01, 0.1), 2)


def bead_contrast(montage, ns_sync, bead):
    """|mean in a 0.6 nm disc at the bead − mean in a ring 2.4–3.6 nm away| on a montage with step ns_sync."""
    px = S_NM / ns_sync                                                   # montage pixel (nm)
    cy, cx = CROP + (bead[1] - RS0) / px, CROP + (bead[2] - RS0) / px      # bead position on the montage grid
    h = int(4 / px) + 2
    y0, x0 = int(cy) - h, int(cx) - h
    patch = montage[y0:y0 + 2 * h + 1, x0:x0 + 2 * h + 1]
    yy, xx = np.indices(patch.shape)
    r = np.hypot(yy + y0 - cy, xx + x0 - cx) * px                          # distance from the bead (nm)
    return abs(patch[r <= 0.6].mean() - patch[(r > 2.4) & (r <= 3.6)].mean())


def depth_scan(data):
    """Contrast of every bead for every N_S, the best N_S per bead, and the depth it implies (nm)."""
    contrast = []
    for ns in NS_GRID:
        montage, _ = shadow_montage(data, ns)
        contrast.append([bead_contrast(montage, ns, b) for b in beads])
    contrast = np.array(contrast)
    best = NS_GRID[contrast.argmax(0)]
    return contrast, best, S_NM / (best * DTHETA) - DF_BEADS


t0 = time.time()
contrast, best_ns, depth_est = depth_scan(data_beads)
print(f"{len(NS_GRID)} montages in {time.time() - t0:.1f} s")
for (z, *_), ns, d in zip(beads, best_ns, depth_est):
    print(f"bead at depth {z:5.0f} nm → best N_S {ns:4.1f} → estimated depth {d:6.0f} nm")
rmse = np.sqrt(np.mean((depth_est - beads[:, 0]) ** 2))
print(f"RMS depth error {rmse:.0f} nm (depth of field λ/α² = {LAMBDA / ALPHA ** 2:.0f} nm)")

### 7.2 How many electrons does it take?

Real cryo specimens tolerate only a few electrons per Å² per image (Module 06). With a 2 nm scan step each probe position
"owns" 400 Å² of specimen, so a dose of 2.5 e⁻/Å² — typical for *one* image of a tilt series — means only 1,000 electrons
per pattern, spread over ~10,000 detector pixels. We redo the depth estimate with **counting noise** (Poisson, Module 04)
at 2.5 and 25 e⁻/Å².

In [ ]:
# ── 7.2 Depth estimates with counting noise ──
results = {"noise-free": (contrast, depth_est)}
for dose in [25.0, 2.5]:
    electrons = dose * (S_NM * 10) ** 2                                   # electrons per pattern (S in Å, squared)
    noisy = np.random.default_rng(5).poisson(data_beads * electrons).astype(np.float32) / electrons
    c_n, _, d_n = depth_scan(noisy)
    results[f"{dose:g} e⁻/Å²"] = (c_n, d_n)
    print(f"dose {dose:4.1f} e⁻/Å² ({electrons:,.0f} electrons per pattern): RMS depth error "
          f"{np.sqrt(np.mean((d_n - beads[:, 0]) ** 2)):.0f} nm")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for j in range(0, len(beads), 2):
    axes[0].plot(NS_GRID, contrast[:, j] / contrast[:, j].max(), label=f"bead at {beads[j, 0]:.0f} nm")
axes[0].set_xlabel("synchronisation step N_S"); axes[0].set_ylabel("bead contrast (normalised)")
axes[0].set_title("each bead comes into focus at its own N_S", fontsize=9); axes[0].legend(fontsize=7)
for (name, (_, d)), m in zip(results.items(), ["o", "s", "^"]):
    axes[1].plot(beads[:, 0], d, m, label=name, alpha=0.8)
axes[1].plot([0, 1250], [0, 1250], "k:", lw=1)
axes[1].set_xlabel("true depth (nm)"); axes[1].set_ylabel("estimated depth (nm)"); axes[1].legend(fontsize=8)
axes[1].set_title("depth from a single scan", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Without noise, a single scan places every bead in depth to within a few tens of nanometres —
far finer than the depth of field (~280 nm), because we measure *where the contrast peaks*, not how wide the peak is.
(The peaks themselves are broad — about one unit of $N_S$, i.e. several hundred nanometres — so two beads stacked
directly above each other would be hard to *separate*: localisation precision is not resolution.) With counting noise the
estimates scatter more: at 25 e⁻/Å² they are nearly as good as without noise; at the 2.5 e⁻/Å² of a single tilt the
error grows about threefold — still finer than the depth of field, but no longer tens of nanometres. Dose is the price of
everything in cryo-EM.

✏️ **Your turn.** Double the convergence angle (`ALPHA = 6e-3`, and halve `PIXEL` to keep the simulation sampled) and
rerun Section 7. The out-of-focus blur $2\alpha\Delta z$ doubles — do the contrast peaks get narrower?

### 7.3 A shadow volume

Stacking the montages for all $N_S$, each resampled onto the same lateral grid, gives a **shadow volume** — a 3D image
from one scan. Viewed from the side, each bead is a vertical streak centred at its depth: the depth resolution is much
coarser than the lateral resolution, just as the missing wedge of Module 07 elongates tomograms along the beam.

In [ ]:
# ── 7.3 Side view of the shadow volume ──
lateral = np.arange(RS0, RS0 + (NSCAN - 1) * S_NM, 0.25)                 # common lateral grid (nm)
volume = []
for ns in NS_GRID:
    m, _ = shadow_montage(data_beads, ns)
    m = m - ndimage.gaussian_filter(m, 8)                                 # remove slow background
    idx = CROP + (lateral - RS0) * ns / S_NM
    volume.append(np.abs(ndimage.map_coordinates(m, np.meshgrid(idx, idx, indexing="ij"), order=1)))
volume = np.array(volume)                                                 # (N_S, y, x)
depth_axis = S_NM / (NS_GRID * DTHETA) - DF_BEADS
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.pcolormesh(lateral, depth_axis, volume.max(1), cmap="magma", shading="auto")
ax.plot(beads[:, 2], beads[:, 0], "c+", ms=10, label="true bead positions")
ax.invert_yaxis(); ax.set_xlabel("x (nm)"); ax.set_ylabel("depth below top layer (nm)"); ax.legend(fontsize=8, loc="lower right")
ax.set_title("shadow volume, side view (maximum over y)", fontsize=9)
plt.tight_layout(); plt.show()

---
## 8 · Towards 4D-STEM tomography

A single scan gives depth, but coarsely. For full 3D, 4D-STEM is combined with **tilting** (Module 07):

- **Cone-beam geometry.** In a montage every detector pixel is a ray from the crossover — not the parallel rays assumed by
  ordinary back-projection. Seifer, Houben & Elbaum (2026) compute a shadow volume for each tilt and back-project these
  with a cone-beam model (the Astra toolbox can do this), followed by 3D deconvolution. Their conclusion: shadow imaging
  relaxes the depth-of-field limit that otherwise restricts STEM tomography of thick specimens to small convergence angles.
- **Contrast for thick cells.** Seifer, Kirchweger, Edel & Elbaum (2024) automated 4D-STEM tilt series (in SerialEM) and
  compared virtual detectors — incoherent and annular bright field, iCOM, PCA of annular detectors — for cryo-tomography,
  reaching individual DNA strands and tail filaments in T4 bacteriophage.
- **tomo4d.** Kiriati, Fernández & Elbaum presented *Tomo4D: 4D-STEM tomographic reconstruction* and *Depth-resolved
  4D-STEM by tomo4d* as conference abstracts in 2026. ⚠️ We could confirm only their titles, authors and meetings; their
  content was not accessible to us, so this notebook does **not** describe the tomo4d method. Section 7 shows the general
  principle the titles point to — depth information inside 4D-STEM data — not the authors' algorithm.

What the toy leaves out: inelastic scattering and multiple scattering in really thick ice; lens aberrations other than
defocus; partial coherence (a finite source size blurs shadows); radiation damage during the scan; detector
imperfections; and the CTF correction of montages.

### 📄 Paper connection — Seifer, Houben & Elbaum 2026: shadow montage and cone-beam reconstruction

**Paper:** Seifer S, Houben L, Elbaum M. *Shadow montage and cone-beam reconstruction in 4D-STEM tomography.* Microscopy
and Microanalysis (2026). [doi:10.1093/mam/ozaf126](https://doi.org/10.1093/mam/ozaf126) (preprint
[bioRxiv 10.1101/2025.09.06.674643](https://doi.org/10.1101/2025.09.06.674643))

- **Question it answers:** can the shadow images inside a defocused-probe 4D-STEM dataset be used for high-resolution,
  depth-resolved imaging and tomography of thick biological specimens?
- **What the paper actually does:** uses a defocused convergent probe so that each far-field pattern is a shadow image
  magnified by ≈ camera length / defocus (conditions: defocus much larger than the depth-of-field unit λ/α² and than the
  spherical-aberration term, crossover consistently above or below the specimen); assembles a **shadow montage** by
  shifting each shadow patch into an upsampled grid by $N_S$ pixels per scan step, summing and normalising by the overlap
  count (underfocus inverts the shadow); applies CTF correction (weak phase, defocus only, contrast inverting between over-
  and underfocus) to the montage; shows that the montage is a back-projection in **cone-beam** geometry and equivalent to
  **tilt-corrected bright field**; uses the dependence of magnification on depth to select layers with $N_S$ (a "shadow
  volume" from one dataset, depth resolution ≈ Δf/N_S); and back-projects per-tilt shadow volumes for tomography, with 3D
  deconvolution.
- **Data reported:** Tecnai T20-F with an ARINA pixelated detector (200 kV, binned patterns, tilts ±60° in 3° steps,
  2–3 e⁻/Å² per tilt) and a Titan Themis; specimens include U2OS, G-8 and MEK cells, T4 phage, and gold on carbon (two
  gold layers sharp at $N_S$ = 16 and 22). MATLAB code on GitHub (`Pr4Et/Supplementaries`, Shadow_Montage).
- **What our toy keeps:** wave-optical shadow formation, the montage with $N_S$, depth selection, the tcBF equivalence,
  the cone-beam picture, the dose question.
- **What our toy leaves out:** real detectors and specimens, CTF correction, tilt series and 3D reconstruction.
- **Limitations stated:** weak-phase assumption, defocus-only correction, the dose/noise trade-off, memory needs of
  cone-beam reconstruction.

### 📄 Paper connection — Seifer, Kirchweger, Edel & Elbaum 2024: optimizing contrast in 4D-STEM cryo-tomography

**Paper:** Seifer S, Kirchweger P, Edel KM, Elbaum M. *Optimizing contrast in automated 4D-STEM cryo-tomography.*
Microscopy and Microanalysis (2024). [doi:10.1093/mam/ozae050](https://doi.org/10.1093/mam/ozae050) (preprint
[bioRxiv 10.1101/2024.02.23.581684](https://doi.org/10.1101/2024.02.23.581684))

- **What it does (from the abstract):** records 4D-STEM tilt series automatically (SerialEM) and reconstructs a
  tomogram for each virtual detector — incoherent and annular bright field, integrated centre of mass, a parallax
  decomposition of simulated iDPC — and uses PCA of virtual annular detectors and 3D deconvolution; resolves DNA strands
  and tail filaments in T4 bacteriophage.
- **In this notebook:** Section 2 (virtual BF/ABF/ADF, COM and iCOM, PCA of rings). ⚠️ We read the abstract, not the
  full paper.

### 📄 Paper connection — Kiriati, Fernández & Elbaum 2026: Tomo4D and depth-resolved 4D-STEM by tomo4d

**Abstracts:** *Tomo4D: 4D-STEM tomographic reconstruction* (89th Annual Meeting of the Israel Chemical Society, Tel Aviv,
February 2026; [hdl:10261/434911](https://hdl.handle.net/10261/434911)) and *Depth-resolved 4D-STEM by tomo4d* (21st
International Microscopy Congress, Liverpool, 2026; [hdl:10261/445318](https://hdl.handle.net/10261/445318)).
⚠️ **Unverified content** — only titles, authors and venues could be confirmed. Read the abstracts themselves before
relying on any description of tomo4d.

---
## ✅ Checkpoint

1. What are the four dimensions of a 4D-STEM dataset?
   <details><summary>Answer</summary>Two scan coordinates (probe position y, x) and two detector coordinates (scattering angles θy, θx): a whole diffraction pattern at every probe position.</details>
2. Why is a weak phase object nearly invisible in an in-focus BF image, but visible in iCOM?
   <details><summary>Answer</summary>A phase shift barely changes the total intensity in the disc; it *deflects* the electrons. The centre of mass of the pattern measures that deflection (the phase gradient), and integrating it recovers the phase.</details>
3. A camera length of 1.5 m and a defocus of 5 µm give what shadow magnification?
   <details><summary>Answer</summary>M = L/Δf = 1.5 m / 5 µm = 300,000×.</details>
4. What does the synchronisation step $N_S$ have to match, and what happens if it doesn't?
   <details><summary>Answer</summary>The shift of the shadow (in montage pixels) per scan step, s/(Δf·δθ). If it doesn't, overlapping patches place a feature at different points and the montage blurs.</details>
5. Why does a deeper layer need a smaller $N_S$?
   <details><summary>Answer</summary>It is farther from the crossover, so its shadow is less magnified and moves by fewer shadow pixels per scan step.</details>
6. Why are shadow montage and tilt-corrected bright field the same?
   <details><summary>Answer</summary>Both add every detector pixel of every pattern into the same grid position; the montage loops over probe positions first, tcBF over detector pixels first.</details>
7. Our beads were located in depth to tens of nm, yet the depth resolution is hundreds of nm. Explain.
   <details><summary>Answer</summary>Precision of locating an isolated peak can be much finer than the peak's width; resolution — separating two nearby objects — is set by the width.</details>

## 🧠 Questions & discussion

1. STEM needs no lens after the specimen. Why does that help for thick specimens? (Think about electrons that lose energy.)
2. 4D-STEM records far more numbers per scan than a conventional image. What does that cost (detector speed, data volume,
   dose per pattern), and what does it buy?
3. Which virtual detector would you choose for a membrane-rich cell, and which for locating gold markers?
4. The depth sectioning of Section 5 works for thin layers. What could go wrong in a continuous, crowded 1 µm specimen?
5. How would you combine depth from one scan with tilting? What would the missing wedge of Module 07 look like in a cone
   beam?

## 🚀 Bonus — going further

- **Underfocus.** Put the crossover below the specimen and build a montage. You will need to flip each patch (`[::-1, ::-1]`) —
  why?
- **CTF correction.** The fringes in the montage are Fresnel phase contrast at defocus Δf. Apply the phase-flipping of
  Module 06 to `m_top` with defocus `DF_TOP` and pixel `S_NM / 10`. Does the vesicle look more like the true phase?
- **Fractional N_S for layers.** Find the $N_S$ that maximises the sharpness of the filaments in `data_two` with a step of
  0.1. Convert it into depth.
- **Real code:** the authors' MATLAB implementation is on GitHub (`Pr4Et/Supplementaries`, folder `Shadow_Montage`).

## 📝 Lab note

```text
Module 08 — date:
Input: simulated thin specimen (focused probe), two-layer and six-layer specimens (defocused probe), 200 kV, α = 3 mrad
Output: virtual BF/ABF/ADF/COM/iCOM images, PCA of rings, shadow montages, focal series, tcBF, bead depths, shadow volume
Method: multislice wave optics, virtual detectors, shadow montage with synchronisation step N_S, parallax
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Seifer, Houben & Elbaum 2026; Seifer et al. 2024; Kiriati, Fernández & Elbaum 2026 (abstracts)
```

## 📚 References

1. Seifer S, Houben L, Elbaum M. Shadow montage and cone-beam reconstruction in 4D-STEM tomography. *Microsc Microanal* (2026). doi:10.1093/mam/ozaf126
2. Seifer S, Kirchweger P, Edel KM, Elbaum M. Optimizing contrast in automated 4D-STEM cryo-tomography. *Microsc Microanal* (2024). doi:10.1093/mam/ozae050
3. Kiriati S, Fernández J-J, Elbaum M. Tomo4D: 4D-STEM tomographic reconstruction. Abstract, 89th Annual Meeting of the Israel Chemical Society (2026). hdl:10261/434911
4. Kiriati S, Fernández J-J, Elbaum M. Depth-resolved 4D-STEM by tomo4d. Abstract, 21st International Microscopy Congress (2026). hdl:10261/445318
5. Wolf SG, Houben L, Elbaum M. Cryo-scanning transmission electron tomography of vitrified cells. *Nat Methods* 11:423–428 (2014). doi:10.1038/nmeth.2842
6. Ophus C. Four-dimensional scanning transmission electron microscopy (4D-STEM): from scanning nanodiffraction to ptychography and beyond. *Microsc Microanal* 25:563–582 (2019). doi:10.1017/S1431927619000497
7. Lazić I, Bosch EGT, Lazar S. Phase contrast STEM for thin samples: integrated differential phase contrast. *Ultramicroscopy* 160:265–280 (2016). doi:10.1016/j.ultramic.2015.10.011

## ➡️ Next

**[09 · Build an in silico cell: crowding, membranes and realistic tomograms](09_in_silico_cell_simulation.ipynb)** — back
to conventional cryo-ET: pack hundreds of molecules into a cell-sized box and push them through the full physics of
Modules 06–07.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")